In [8]:
import pandas as pd
from openai import OpenAI
import os

In [9]:
api_key=os.environ["OPENAI_API_KEY"]

In [10]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter #This split by new line
from langchain_text_splitters import RecursiveCharacterTextSplitter  #This split by character not necessarily by new line i.e \n

#Third example of creating and splitting documents

url="https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df=pd.read_csv(url)
#display(df)

"""df = pd.DataFrame({
    "Name": ["Lizy", "Methew", "Paul"],
    "gender": ["Female", "Female", "Male"]
})
"""
doc3=[Document(page_content=", ".join(f"{col}: {row[col]}" for col in df.columns) , metadata={"row": i+1}) for i, row in df.iterrows()]

text_splitter3=CharacterTextSplitter(chunk_size=50, chunk_overlap=10)
chunks3=text_splitter3.split_documents(doc3)


In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()  #convert text into numerical values, using openAI trained data for vector embeddings
vectorstore= FAISS.from_documents(chunks3,embeddings)
retriever= vectorstore.as_retriever()

In [5]:
#This is building a RAG pipeline using LCEL (LangChain Expression Language),
#basically a clean, composable way to define how data flows through your system.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

"""User Question
     ↓
Pass through + Retrieve context
     ↓
Insert into prompt
     ↓
LLM generates answer
     ↓
Convert to clean string"""

#Creatiing LLMs

llm = ChatOpenAI(model="gpt-4.1", temperature=0)


prompt= PromptTemplate.from_template(
    """Answer the following question:
    {question}
    Based on the following context:
    {context}
    """
)

chain= (
    {"context": retriever, "question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)


In [6]:
from langchain_core.runnables import RunnableWithMessageHistory
#from langchain.memory import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

#Define a second prompt which accomodates chat history


prompt2= PromptTemplate.from_template("""
You are an assistant.

A pandas DataFrame {context} is called df already exists.

chat history:
{chat_history}

User question:
{question}

Return ONLY a valid Python dictionary (not JSON string, not markdown).

The dictionary must have EXACTLY these keys:
- "text"
- "dataframe_code"
- "plot_code"

Format:
{{
  "text": "...",
  "dataframe_code": "...",
  "plot_code": "..."
}}

Rules:

TEXT:
- Only plain explanation
- No code


DATAFRAME_CODE:
- Write pandas code that uses the most recent available dataframe
- If a previous result_df exists, use it; otherwise use df
- DO NOT recreate df
- DO NOT define data = [...]
- DO NOT import pandas
- ALWAYS output updated dataframe as result_df
- Use pd.concat() instead of df.append()


PLOT_CODE:
- ONLY Plotly code
- Use plotly.express as px (assume it is already imported as px)
- Create a figure using px.*
- Store the figure in a variable called fig
- DO NOT call fig.show()
- Use result_df if available, otherwise df
- DO NOT import plotly
- DO NOT include explanations

- ONLY generate ONE Plotly figure stored as fig
- If the user requests multiple plots or multiple variables, choose ONLY the single most relevant plot

- ONLY if user explicitly asks for chart/graph/plot; use Plotly (px) and store fig; otherwise "None"; never plot for explanations.

DO NOT use px.table (it does not exist)
If a table is requested, use plotly.graph_objects.Table instead


GLOBAL RULES:
- If dataframe is not needed → "dataframe_code": "None"
- If plot is not needed → "plot_code": "None"
- Do NOT change key names
- Do NOT add extra keys
- Output must be directly usable as a Python dictionary
- Retrieved context is NOT a dataframe
""")

#Using a new cjhain to accomaodate the chat history inside an input mapping
chain2 = (
    {
        "context": itemgetter("question") | retriever | (lambda docs: "\n".join(d.page_content for d in docs)),
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt2
    | llm
    | StrOutputParser()
)

#define a dictionary that will contain a message history

store={}

def get_message_history(session_id:str):
    if session_id not in store:
        store[session_id]= InMemoryChatMessageHistory()
    return store[session_id]


#wrap the chain2 with RunnableWithMessageHistory, to get chai with memory

chain_with_memory= RunnableWithMessageHistory(
    chain2,
    get_message_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

#Invoke with memory

response2 = chain_with_memory.invoke(
    {"question":"explain whay is in the data"},
    config={"configurable":{"session_id":"user1"}}
)

print(response2)

{
  "text": "The data contains information about Titanic passengers, including their ID, survival status, ticket class, name, gender, age, number of siblings/spouses and parents/children aboard, ticket number, fare paid, cabin, and port of embarkation.",
  "dataframe_code": "None",
  "plot_code": "None"
}


In [12]:
import json
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
#Invoke with memory

leave=["quit","exit"]
while True:

    user_input= input("user: ")

    if user_input.lower() in leave:
        break
        
    response3 = chain_with_memory.invoke(
        {"question":user_input},
        config={"configurable":{"session_id":"user1"}}
    )

    final_answer=json.loads(response3)  #We need to ensure that the answer is a dictionary 
    
    text=final_answer['text']
    print("\n Assistant: \n"+str(text))

    dataframe_code=final_answer['dataframe_code']
    plot_code=final_answer['plot_code']

    #create local vars
    glob_vars={"df": df, "pd":pd, "np":np,"px":px, "result_df": result_df}
    # Evaluate the code to get the resulting DataFrame
    exec(dataframe_code, glob_vars)
    result_df= glob_vars.get("result_df")

    if dataframe_code != "None" and plot_code == "None":
        try:
            display(result_df)
            df=result_df.copy()
                
        except Exception as e:
            print("Error executing dataframe code:", e)
  
    if plot_code != "None":
        try:
            #add local variable
            local_vars={"df": df, "np":np,"result_df": result_df,"px":px}
            # Execute the plotting code with df and plt in scope
            exec(plot_code, {},local_vars )

            fig = local_vars.get("fig")
            print(fig.to_json())
            if fig:
                fig.show("notebook")
                
        except Exception as e:
            print("Error executing plot code:", e)

    

user:  explain the data


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}